# UCSF-PDGM clean study: clinical only, MRI only, multimodal, and missing clinical data
Run the cells from top to bottom (Runtime > Run all also works). Order of work:
1. **BINARY** (grade 2/3 vs grade 4): clinical only, MRI only, MRI + clinical, MRI + clinical with missing clinical data.
2. **3-CLASS** (grade 2 / 3 / 4): the same four parts.
3. A final report and result files are written at the end.

To use other models, edit the **SETTINGS** cell only (see the comments there).

# **0. SETUP**

In [ ]:
# Install the libraries (Colab already has most of them)
!pip -q install datasets scikit-learn pandas matplotlib openpyxl torch torchvision

In [ ]:
import os, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
warnings.filterwarnings("ignore")

# **1. LOAD DATA**
One row = one patient (one tumour slice, 4 MRI sequences, age, sex, grade).

In [ ]:
# 1) Dataset load aur inspect
from datasets import load_dataset, Image as HFImage

ds = load_dataset("MedOtter/UCSF-PDGM")
print(ds)
split = list(ds.keys())[0]
d = ds[split]
print("split:", split, "| rows:", len(d))
img_cols = [k for k, v in d.features.items() if isinstance(v, HFImage)]
print("image columns:", img_cols)
print("other columns:", [k for k in d.features if k not in img_cols])

In [ ]:
# 2) Metadata (images ke baghair) aur sahi columns dhoondo
import re, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

meta = d.remove_columns(img_cols).to_pandas()
print(meta.shape)
print(meta.head(3).T)

ID_COL = next(c for c in meta.columns
              if meta[c].astype(str).str.match(r"^UCSF-PDGM-\d+").mean() > 0.9)
GRADE_COL = next(c for c in meta.columns if "grade" in c.lower() and "brats" not in c.lower())
AGE_COL = next(c for c in meta.columns if c.lower().startswith("age"))
SEX_COL = next(c for c in meta.columns if c.lower().startswith("sex"))
print("Using:", dict(id=ID_COL, grade=GRADE_COL, age=AGE_COL, sex=SEX_COL))

In [ ]:
# 3) Patient table (ek row = ek patient) aur label
patients = pd.DataFrame({
    "row": np.arange(len(d)),
    "patient_id": meta[ID_COL].astype(str),
    "sex": meta[SEX_COL].astype(str).str.strip().str.upper().str[0],
    "age": pd.to_numeric(meta[AGE_COL].astype(str).str.extract(r"(\d+\.?\d*)")[0], errors="coerce"),
    "who_grade": pd.to_numeric(meta[GRADE_COL].astype(str).str.extract(r"(\d)")[0], errors="coerce"),
})
patients["sex_m"] = patients["sex"].map({"M": 1.0, "F": 0.0})        # M=1, F=0, baaki NaN

print("duplicate patient ids:", patients["patient_id"].duplicated().sum())
patients = patients.drop_duplicates("patient_id").reset_index(drop=True)
print("grade counts:", patients["who_grade"].value_counts(dropna=False).to_dict())
print("sex counts:", patients["sex"].value_counts(dropna=False).to_dict())
print("age missing:", patients["age"].isna().sum(), "| age range:", patients["age"].min(), "-", patients["age"].max())

patients = patients.dropna(subset=["who_grade"]).reset_index(drop=True)
patients["y"] = (patients["who_grade"] == 4).astype(int)            # grade 4 -> 1, grade 2/3 -> 0
print("used patients:", len(patients), "| grade 4:", int(patients["y"].sum()),
      "| grade 2/3:", int((patients["y"] == 0).sum()),
      "| always-grade-4 accuracy:", round(patients["y"].mean(), 3))

# Leakage guard: sirf age aur sex features hain
CLINICAL_FEATURES = ["age", "sex_m"]

# **2. MRI NUMBERS** (frozen ResNet50, no training)
The numbers are saved in `embeddings.npz`, so this step runs only once.

In [ ]:
# 6a) Frozen ResNet50 (ImageNet) se features. Koi training nahi, sirf forward pass.
import torch, torchvision
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights

device = "cuda" if torch.cuda.is_available() else "cpu"
_model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
_model.fc = torch.nn.Identity()
_model = _model.eval().to(device)
_tf = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(),
                          transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

@torch.no_grad()
def extract_embeddings(pil_images, batch_size=32):
    out = []
    for i in range(0, len(pil_images), batch_size):
        x = torch.stack([_tf(im.convert("RGB")) for im in pil_images[i:i + batch_size]]).to(device)
        out.append(_model(x).cpu().numpy())
    return np.concatenate(out)

In [ ]:
# 6b) Har sequence ke liye embeddings nikaalo aur save karo (dobara nahi chalana padega)
import os
SEQS = [c for c in img_cols if c.lower() in ("t1", "t1c", "t2", "flair")]
print("sequences used:", SEQS)

if os.path.exists("embeddings.npz"):
    emb = dict(np.load("embeddings.npz"))
    print("loaded saved embeddings")
else:
    emb = {}
    rows = patients["row"].astype(int).tolist()
    for s in SEQS:
        dseq = d.select_columns([s])
        imgs = [dseq[i][s] for i in rows]
        emb[s] = extract_embeddings(imgs)
        print(s, emb[s].shape)
    np.savez("embeddings.npz", **emb)
assert all(emb[s].shape[0] == len(patients) for s in SEQS)

# **3. SETTINGS AND MODEL CHOICES**

In [ ]:
# ================= SETTINGS: change things only here =================
SEEDS   = [42, 43, 44]      # the whole study is repeated 3 times with different patient splits
N_FOLDS = 5                 # patient-level 5-fold cross-validation
N_PCA   = 32                # the MRI numbers of each patient are reduced to 32 numbers
MRI_SEQUENCES = ["T1", "T1c", "T2", "FLAIR"]   # use ["T1c"] to try one sequence only
TRAIN_WITH_MISSING = True  # Train on complete data; optionally add randomly missing clinical copies
TAG = "deep_with_missing"  # Unique output tag for this deep-learning experiment

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC

# ---- Model recipes. Each function returns a NEW untrained model. The function name is used as the model name in all tables ----
def logistic_regression(): return LogisticRegression(C=0.1, class_weight="balanced", max_iter=3000)
def decision_tree():       return DecisionTreeClassifier(max_depth=4, min_samples_leaf=5, class_weight="balanced", random_state=0)
def random_forest():       return RandomForestClassifier(n_estimators=200, min_samples_leaf=3, class_weight="balanced", random_state=0, n_jobs=-1)
# Two more ready recipes (NOT used until you add their names to the lists below):
def svm():                       return SVC(C=1.0, kernel="rbf", probability=True, class_weight="balanced", random_state=0)
def hist_gradient_boosting():    return HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=150, class_weight="balanced", random_state=0)
# Example: MULTIMODAL_MODELS = [logistic_regression, decision_tree, random_forest, svm, hist_gradient_boosting]
# You can write your own the same way. Every recipe must be its own "def" so that it has a name.

# ---- Which models are used in each part. To replace a model, change its name in the list (the tables relabel themselves) ----
#MRI_MODELS        = [logistic_regression, decision_tree, random_forest]
#CLINICAL_MODELS   = [logistic_regression, decision_tree, random_forest]
#MULTIMODAL_MODELS = [logistic_regression, decision_tree, random_forest]
MRI_MODELS        = [logistic_regression, decision_tree, random_forest, svm, hist_gradient_boosting]
CLINICAL_MODELS   = [logistic_regression, decision_tree, random_forest, svm, hist_gradient_boosting]
MULTIMODAL_MODELS = [logistic_regression, decision_tree, random_forest, svm, hist_gradient_boosting]

def model_name(make_model):
    name = make_model.__name__.replace("_", " ").title()
    return {"Svm": "SVM", "Knn": "KNN", "Xgboost": "XGBoost"}.get(name, name)

# ---- How much clinical data is hidden in the missing-data part (test patients only) ----
MISSING_SCENARIOS = ["0% missing", "25% missing", "50% missing", "75% missing", "100% missing",
                     "only age missing", "only sex missing"]

# ---- Main metric of each task (used to pick the "best" model) ----
PRIMARY = {"binary": "mcc", "3class": "macro_f1"}

# **4. HELPER FUNCTIONS AND THE FOUR PARTS**

In [ ]:
# ================= HELPER FUNCTIONS (you do not need to change these) =================
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             matthews_corrcoef, roc_auc_score, recall_score)

age_all = patients["age"].values.astype(float)
sex_all = patients["sex_m"].values.astype(float)               # M = 1, F = 0
X_mri_all = np.concatenate([emb[s] for s in MRI_SEQUENCES], axis=1)   # each MRI sequence gives 2048 numbers per patient

def get_folds(y):
    """Yield (seed, fold, train_idx, test_idx). One row = one patient, so every split is patient-level."""
    for seed in SEEDS:
        for k, (tr, te) in enumerate(StratifiedKFold(N_FOLDS, shuffle=True, random_state=seed).split(X_mri_all, y)):
            yield seed, k, tr, te

def mri_features(tr, te):
    """MRI numbers -> 32 numbers per patient. Learned on the training patients only."""
    scaler = StandardScaler().fit(X_mri_all[tr])
    pca = PCA(n_components=min(N_PCA, len(tr) - 1), whiten=True, random_state=0).fit(scaler.transform(X_mri_all[tr]))
    to_features = lambda idx: pca.transform(scaler.transform(X_mri_all[idx]))
    return to_features(tr), to_features(te)

def clinical_stats(tr):
    """Average age and most common sex of the training patients (used to fill missing values)."""
    return dict(age_mean=np.nanmean(age_all[tr]), age_sd=np.nanstd(age_all[tr]) or 1.0,
                sex_mode=float(pd.Series(sex_all[tr]).mode().iloc[0]))

def clinical_features(age, sex, stats):
    """Clinical table with 4 columns: age (scaled), sex, age_is_missing, sex_is_missing."""
    age_missing, sex_missing = np.isnan(age), np.isnan(sex)
    age_filled = np.where(age_missing, stats["age_mean"], age)
    sex_filled = np.where(sex_missing, stats["sex_mode"], sex)
    return np.column_stack([(age_filled - stats["age_mean"]) / stats["age_sd"], sex_filled,
                            age_missing.astype(float), sex_missing.astype(float)])

def hide_clinical(age, sex, scenario, rng):
    """Remove clinical values to simulate missing data."""
    age, sex = age.copy(), sex.copy()
    if scenario == "only age missing":
        age[:] = np.nan
    elif scenario == "only sex missing":
        sex[:] = np.nan
    else:                                   # for example "50% missing": that share of patients loses BOTH age and sex
        share = int(scenario.split("%")[0]) / 100
        chosen = rng.choice(len(age), int(round(share * len(age))), replace=False)
        age[chosen] = np.nan
        sex[chosen] = np.nan
    return age, sex

def hide_randomly(age, sex, rng, p=0.4):
    """Used only when TRAIN_WITH_MISSING = True: hide each value with probability p."""
    age, sex = age.copy(), sex.copy()
    age[rng.random(len(age)) < p] = np.nan
    sex[rng.random(len(sex)) < p] = np.nan
    return age, sex

def score(model_name, seed, k, y_true, proba, class_names, **extra):
    """All metrics of one model on one test fold."""
    n = len(class_names)
    assert proba.shape[1] == n, "model output does not match the number of classes"
    pred = proba.argmax(axis=1)
    try:
        auc = roc_auc_score(y_true, proba[:, 1]) if n == 2 else \
              roc_auc_score(y_true, proba, multi_class="ovr", average="macro", labels=list(range(n)))
    except ValueError:
        auc = np.nan
    row = dict(model=model_name, seed=seed, fold=k, mcc=matthews_corrcoef(y_true, pred),
               balanced_acc=balanced_accuracy_score(y_true, pred), macro_f1=f1_score(y_true, pred, average="macro"),
               auc=auc, accuracy=accuracy_score(y_true, pred))
    for name, r in zip(class_names, recall_score(y_true, pred, average=None, labels=list(range(n)))):
        row["recall " + name] = r
    row.update(extra)
    return row

In [ ]:
# ================= THE FOUR PARTS OF THE STUDY =================

def run_clinical_only(y, class_names):
    """PART 1: only clinical data (age, sex)."""
    rows = []
    for seed, k, tr, te in get_folds(y):
        stats = clinical_stats(tr)
        C_tr = clinical_features(age_all[tr], sex_all[tr], stats)
        C_te = clinical_features(age_all[te], sex_all[te], stats)
        for make_model in CLINICAL_MODELS:
            name = model_name(make_model)
            model = make_model().fit(C_tr, y[tr])
            rows.append(score(name, seed, k, y[te], model.predict_proba(C_te), class_names))
    return pd.DataFrame(rows)

def run_mri_only(y, class_names):
    """PART 2: only MRI (4 sequences)."""
    rows = []
    for seed, k, tr, te in get_folds(y):
        Z_tr, Z_te = mri_features(tr, te)
        for make_model in MRI_MODELS:
            name = model_name(make_model)
            model = make_model().fit(Z_tr, y[tr])
            rows.append(score(name, seed, k, y[te], model.predict_proba(Z_te), class_names))
    return pd.DataFrame(rows)

def run_multimodal(y, class_names):
    """PART 3: MRI + clinical data together (the two sets of numbers are placed side by side)."""
    rows = []
    for seed, k, tr, te in get_folds(y):
        Z_tr, Z_te = mri_features(tr, te)
        stats = clinical_stats(tr)
        X_tr = np.hstack([Z_tr, clinical_features(age_all[tr], sex_all[tr], stats)])
        X_te = np.hstack([Z_te, clinical_features(age_all[te], sex_all[te], stats)])
        for make_model in MULTIMODAL_MODELS:
            name = model_name(make_model)
            model = make_model().fit(X_tr, y[tr])
            rows.append(score(name, seed, k, y[te], model.predict_proba(X_te), class_names))
    return pd.DataFrame(rows)

def run_missing_clinical(y, class_names):
    """PART 4: the same trained models are tested while clinical data is hidden (MRI-only is included as a reference)."""
    rows = []
    for seed, k, tr, te in get_folds(y):
        Z_tr, Z_te = mri_features(tr, te)
        stats = clinical_stats(tr)
        age_fit, sex_fit, Z_fit, y_fit = age_all[tr], sex_all[tr], Z_tr, y[tr]
        if TRAIN_WITH_MISSING:               # add a second copy of the training patients with random missing values
            age2, sex2 = hide_randomly(age_all[tr], sex_all[tr], np.random.default_rng(seed * 100 + k))
            age_fit, sex_fit = np.concatenate([age_fit, age2]), np.concatenate([sex_fit, sex2])
            Z_fit, y_fit = np.vstack([Z_fit, Z_tr]), np.concatenate([y_fit, y[tr]])
        C_fit = clinical_features(age_fit, sex_fit, stats)

        trained = {}                          # train every model once
        for make_model in MRI_MODELS:
            trained["MRI - " + model_name(make_model)] = ("mri", make_model().fit(Z_fit, y_fit))
        for make_model in CLINICAL_MODELS:
            trained["Clinical - " + model_name(make_model)] = ("clinical", make_model().fit(C_fit, y_fit))
        for make_model in MULTIMODAL_MODELS:
            trained["MRI+Clinical - " + model_name(make_model)] = ("both", make_model().fit(np.hstack([Z_fit, C_fit]), y_fit))

        for i, scenario in enumerate(MISSING_SCENARIOS):
            rng = np.random.default_rng(seed * 1000 + k * 10 + i)      # same hidden patients for every model
            age_te, sex_te = hide_clinical(age_all[te], sex_all[te], scenario, rng)
            C_te = clinical_features(age_te, sex_te, stats)
            for name, (kind, model) in trained.items():
                if kind == "mri":        X_te = Z_te
                elif kind == "clinical": X_te = C_te
                else:                    X_te = np.hstack([Z_te, C_te])
                rows.append(score(name, seed, k, y[te], model.predict_proba(X_te), class_names, scenario=scenario))
    out = pd.DataFrame(rows)
    out["scenario"] = pd.Categorical(out["scenario"], MISSING_SCENARIOS, ordered=True)
    return out

In [ ]:
# ================= TABLES, FIGURE, FILES, REPORT =================

def summarize(df, by):
    """Mean and standard deviation over all seeds and folds."""
    cols = [c for c in df.columns if c not in ("model", "scenario", "seed", "fold")]
    grouped = df.groupby(by, observed=True)[cols]
    return grouped.mean(), grouped.std()

def show_columns(df):
    main = [c for c in ["mcc", "balanced_acc", "macro_f1", "auc", "accuracy"] if c in df.columns]
    return main + [c for c in df.columns if c.startswith("recall ")]

def pretty(mean, sd, cols):
    out = pd.DataFrame(index=mean.index)
    for c in cols:
        out[c] = [f"{m:.3f} ± {s:.3f}" for m, s in zip(mean[c], sd[c])]
    return out

def show(title, df):
    """Print one results table (one row per model)."""
    mean, sd = summarize(df, ["model"])
    display(Markdown(f"### **{title}**"))
    display(pretty(mean, sd, show_columns(df)))

def show_missing(title, df, task):
    """Print the missing-data results: rows = models, columns = how much clinical data is missing."""
    mean, sd = summarize(df, ["model", "scenario"])
    display(Markdown(f"### **{title}**"))
    for metric in [PRIMARY[task], "balanced_acc"]:
        display(Markdown(f"**{metric}** (mean ± sd over seeds and folds)"))
        display(pretty(mean, sd, [metric])[metric].unstack("scenario"))

def plot_missing(df, task, filename):
    """Line plot: main metric against the percentage of patients with missing clinical data (best model of each kind)."""
    primary = PRIMARY[task]
    levels = [s for s in MISSING_SCENARIOS if s.endswith("% missing")]
    mean = df[df["scenario"].isin(levels)].groupby(["model", "scenario"], observed=True)[primary].mean().unstack()[levels]
    plt.figure(figsize=(7, 4.5))
    for kind in ["MRI - ", "Clinical - ", "MRI+Clinical - "]:
        part = mean[mean.index.str.startswith(kind)]
        best = part["0% missing"].idxmax()
        plt.plot([int(s.split("%")[0]) for s in levels], part.loc[best].values, marker="o", label=best)
    plt.xlabel("Patients with clinical data removed (%)"); plt.ylabel(primary)
    plt.title(f"Effect of missing clinical data ({task})"); plt.legend(fontsize=8); plt.grid(alpha=.3)
    plt.tight_layout(); plt.savefig(filename, dpi=200); plt.show()

PART_SHEETS = [("1 Clinical only", "clinical", ["model"]), ("2 MRI only", "mri", ["model"]),
               ("3 MRI+Clinical", "multimodal", ["model"]), ("4 Missing clinical", "missing", ["model", "scenario"])]

def save_task(task, res):
    """One Excel file per task: one sheet per part (each model has its own row) + raw results of every fold."""
    with pd.ExcelWriter(f"results_{task}_{TAG}.xlsx") as xl:
        for sheet, key, by in PART_SHEETS:
            mean, sd = summarize(res[key], by)
            pretty(mean, sd, show_columns(res[key])).to_excel(xl, sheet_name=sheet)
        pd.concat([d.assign(part=k) for k, d in res.items()]).to_excel(xl, sheet_name="raw folds", index=False)

def md_table(df):
    header = "| model | " + " | ".join(df.columns) + " |"
    lines = [header, "|" + "---|" * (len(df.columns) + 1)]
    lines += ["| " + str(i) + " | " + " | ".join(r) + " |" for i, r in zip(df.index, df.values.tolist())]
    return "\n".join(lines)

def task_report(task, title, y, class_names, res):
    primary = PRIMARY[task]
    pname = {"mcc": "MCC", "macro_f1": "macro-F1"}[primary]
    counts = ", ".join(f"{n}: {int((y == i).sum())}" for i, n in enumerate(class_names))
    L = [f"## {title}", "", f"Patients: {len(y)} ({counts}). Main metric: {pname}. "
         f"Every number is mean ± standard deviation over {len(SEEDS)} seeds x {N_FOLDS} patient-level folds.", ""]
    best_mri = None
    for heading, key in [("Part 1. Clinical only (age, sex)", "clinical"), ("Part 2. MRI only", "mri"),
                         ("Part 3. MRI + clinical (multimodal)", "multimodal")]:
        mean, sd = summarize(res[key], ["model"])
        best = mean[primary].idxmax()
        if key == "mri": best_mri = (best, mean.loc[best, primary])
        L += [f"### {heading}", f"Models used: {', '.join(mean.index)}.",
              f"Best by {pname}: **{best}** ({mean.loc[best, primary]:.3f}).", "",
              md_table(pretty(mean, sd, show_columns(res[key]))), ""]
    mean, sd = summarize(res["missing"], ["model", "scenario"])
    table = pretty(mean, sd, [primary])[primary].unstack("scenario")[MISSING_SCENARIOS]
    L += ["### Part 4. MRI + clinical with missing clinical data", f"Table: {pname} for each model and each amount of missing clinical data.", "",
          md_table(table), "", "Effect of missing clinical data on each multimodal model:", ""]
    for name in [m for m in mean.index.get_level_values("model").unique() if m.startswith("MRI+Clinical")]:
        full, none = mean.loc[(name, "0% missing"), primary], mean.loc[(name, "100% missing"), primary]
        spread = sd.loc[(name, "0% missing"), primary]
        note = "inside the fold-to-fold spread" if abs(none - full) <= spread else "larger than the fold-to-fold spread"
        L.append(f"- {name}: {pname} {full:.3f} with full clinical data and {none:.3f} with all clinical data missing "
                 f"(change {none - full:+.3f}, {note}). Only age missing: {mean.loc[(name, 'only age missing'), primary]:.3f}; "
                 f"only sex missing: {mean.loc[(name, 'only sex missing'), primary]:.3f}.")
    clin = [m for m in mean.index.get_level_values("model").unique() if m.startswith("Clinical")]
    L += ["", "Reference points with all clinical data missing: "
          + "; ".join(f"{m} {mean.loc[(m, '100% missing'), primary]:.3f}" for m in clin)
          + f"; best MRI-only model ({best_mri[0]}) {best_mri[1]:.3f}.", ""]
    return "\n".join(L)

PREVIOUS_WORK = """## Previous work and earlier runs

- Kheirbakhsh et al. (2026, medRxiv preprint, not peer reviewed): UCSF-PDGM, 495 patients, MRI + age + sex, grade 2&3 vs 4, slice-level evaluation with fine-tuned DenseNet121. Clinical-only: accuracy 0.80, MCC 0.44. Best single MRI sequence (T1c): accuracy 0.83, MCC 0.48. Best fusion: accuracy 0.86, MCC 0.54. Check the numbers in the PDF before citing. A missing-clinical-data test was not seen in that paper.
- Earlier run in this project (notebook 04, binary task, frozen ResNet50 features, mean over 3 seeds x 5 folds): MRI-only MCC 0.392; clinical-only MCC 0.451; concatenation fusion MCC 0.447; with all clinical data removed fusion MCC 0.361, MRI-only 0.392, clinical-only 0.000.
- Earlier scratch-CNN baseline (tumor.ipynb, 3 classes, MRI only, one split of 75 test patients): accuracy 0.8133, balanced accuracy 0.4019, macro-F1 0.3978; almost every patient was predicted as grade 4.

These earlier numbers use other models and settings, so compare only the direction of the effects."""

LIMITS = """## How to read these results

- No significance tests are included here. Differences smaller than the standard deviation should not be treated as real.
- One tumour slice per patient, MRI described by frozen ResNet50 features (no fine-tuning), only age and sex as clinical data.
- Missing clinical data is simulated by hiding values; it is not real hospital missingness.
- Grade 2 and grade 3 have few patients, so three-class results (especially their per-class recall) are unstable.
- Dataset: UCSF-PDGM (Calabrese et al., 2022), Hugging Face copy MedOtter/UCSF-PDGM (CC BY 4.0). Cite the original dataset."""

def make_report(runs):
    """Write final_report.md: models used, results of every part, effect of missing clinical data, earlier work."""
    names = lambda models: ", ".join(model_name(m) for m in models)
    models = (f"MRI models: {names(MRI_MODELS)}. Clinical models: {names(CLINICAL_MODELS)}. "
              f"Multimodal models: {names(MULTIMODAL_MODELS)}. Training with missing values: {TRAIN_WITH_MISSING}.")
    parts = ["# Final report: MRI, clinical and multimodal glioma grade classification (UCSF-PDGM)", "",
             "## What was done", "",
             f"- Data: {len(patients)} patients, one tumour slice per patient, 4 MRI sequences (T1, T1c, T2, FLAIR) and age + sex.",
             "- MRI features: frozen ResNet50 (ImageNet) numbers, reduced with PCA.",
             f"- {models}",
             f"- Evaluation: {N_FOLDS}-fold patient-level cross-validation repeated with seeds {SEEDS}.",
             "- Order of work: binary task first (grade 4 vs grade 2/3), then three classes (grade 2 / 3 / 4). "
             "Each task has four parts: clinical only, MRI only, MRI + clinical, MRI + clinical with missing clinical data.", ""]
    for task, title, y, names, res in runs:
        parts.append(task_report(task, title, y, names, res))
    parts += [PREVIOUS_WORK, "", LIMITS, ""]
    text = "\n".join(parts)
    with open(f"final_report_{TAG}.md", "w", encoding="utf-8") as f:
        f.write(text)
    return text

# **BINARY CLASSIFICATION: grade 2/3 vs grade 4**

## **PART 1: CLINICAL ONLY**

In [ ]:
y_bin = (patients["who_grade"] == 4).astype(int).values      # grade 4 = 1, grade 2 or 3 = 0
names_bin = ["grade 2/3", "grade 4"]
binary = {}

binary["clinical"] = run_clinical_only(y_bin, names_bin)
show("BINARY | PART 1: CLINICAL ONLY (age, sex)", binary["clinical"])

## **PART 2: MRI ONLY**
Every MRI model has its own row.

In [ ]:
binary["mri"] = run_mri_only(y_bin, names_bin)
show("BINARY | PART 2: MRI ONLY", binary["mri"])

## **PART 3: MRI + CLINICAL (multimodal)**

In [ ]:
binary["multimodal"] = run_multimodal(y_bin, names_bin)
show("BINARY | PART 3: MRI + CLINICAL (multimodal)", binary["multimodal"])

## **PART 4: MRI + CLINICAL WITH MISSING CLINICAL DATA**

In [ ]:
binary["missing"] = run_missing_clinical(y_bin, names_bin)
show_missing("BINARY | PART 4: MRI + CLINICAL WITH MISSING CLINICAL DATA", binary["missing"], "binary")
plot_missing(binary["missing"], "binary", f"binary_missing_plot_{TAG}.png")

# **3-CLASS CLASSIFICATION: grade 2 / grade 3 / grade 4**

## **PART 1: CLINICAL ONLY**

In [ ]:
y_3c = patients["who_grade"].map({2: 0, 3: 1, 4: 2}).astype(int).values   # grade 2 = 0, grade 3 = 1, grade 4 = 2
names_3c = ["grade 2", "grade 3", "grade 4"]
three = {}

three["clinical"] = run_clinical_only(y_3c, names_3c)
show("3-CLASS | PART 1: CLINICAL ONLY (age, sex)", three["clinical"])

## **PART 2: MRI ONLY**

In [ ]:
three["mri"] = run_mri_only(y_3c, names_3c)
show("3-CLASS | PART 2: MRI ONLY", three["mri"])

## **PART 3: MRI + CLINICAL (multimodal)**

In [ ]:
three["multimodal"] = run_multimodal(y_3c, names_3c)
show("3-CLASS | PART 3: MRI + CLINICAL (multimodal)", three["multimodal"])

## **PART 4: MRI + CLINICAL WITH MISSING CLINICAL DATA**

In [ ]:
three["missing"] = run_missing_clinical(y_3c, names_3c)
show_missing("3-CLASS | PART 4: MRI + CLINICAL WITH MISSING CLINICAL DATA", three["missing"], "3class")
plot_missing(three["missing"], "3class", f"3class_missing_plot_{TAG}.png")

# **5. SAVE RESULTS AND WRITE THE FINAL REPORT**

In [ ]:
# Save everything: two Excel files (one per task), two plots (already saved) and the final report
save_task("binary", binary)
save_task("3class", three)
report = make_report([("binary", "Binary classification: grade 2/3 vs grade 4", y_bin, names_bin, binary),
                      ("3class", "Three-class classification: grade 2 / grade 3 / grade 4", y_3c, names_3c, three)])
display(Markdown(report))
print(f"Saved: results_binary_{TAG}.xlsx, results_3class_{TAG}.xlsx, binary_missing_plot_{TAG}.png, 3class_missing_plot_{TAG}.png, final_report_{TAG}.md")
# On Colab you can download them with:
# from google.colab import files
# for f in [f"final_report_{TAG}.md", f"results_binary_{TAG}.xlsx", f"results_3class_{TAG}.xlsx"]: files.download(f)

# Frozen ResNet50 and EfficientNet-B0 experiments

This extension adds two frozen ImageNet feature extractors with a trainable linear classification head:
- Frozen ResNet50 + linear head
- Frozen EfficientNet-B0 + linear head

The CNN backbones are frozen and used only to extract features. The linear classification head is trained inside each training fold. MRI-only and MRI+clinical fusion are evaluated for binary and three-class tasks. Clinical-only is not applicable to image backbones.

Run this notebook from top to bottom. The first run needs internet access to load the Hugging Face dataset and pretrained EfficientNet weights. A GPU is recommended. No new performance results are claimed until the notebook finishes running.


In [ ]:
# ================= FROZEN EFFICIENTNET-B0 FEATURE EXTRACTION =================
# The backbone is pretrained on ImageNet and frozen. Only a forward pass is used here.
import os
import numpy as np
import torch
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

DEVICE_DEEP = "cuda" if torch.cuda.is_available() else "cpu"
EFF_CACHE = "efficientnet_b0_embeddings.npz"

_eff_weights = EfficientNet_B0_Weights.IMAGENET1K_V1
_eff_model = efficientnet_b0(weights=_eff_weights)
_eff_model.classifier = torch.nn.Identity()
_eff_model.eval().to(DEVICE_DEEP)

_eff_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

@torch.no_grad()
def extract_effnet_embeddings(pil_images, batch_size=32):
    outputs = []
    for start in range(0, len(pil_images), batch_size):
        batch_images = pil_images[start:start + batch_size]
        batch = torch.stack([_eff_tf(im.convert("RGB")) for im in batch_images]).to(DEVICE_DEEP)
        outputs.append(_eff_model(batch).cpu().numpy())
    return np.concatenate(outputs, axis=0)

if os.path.exists(EFF_CACHE):
    eff_emb = dict(np.load(EFF_CACHE))
    print("Loaded cached EfficientNet-B0 embeddings:", {k: v.shape for k, v in eff_emb.items()})
else:
    eff_emb = {}
    _rows = patients["row"].astype(int).tolist()
    for _seq in MRI_SEQUENCES:
        _seq_images = [d[int(_row)][_seq] for _row in _rows]
        eff_emb[_seq] = extract_effnet_embeddings(_seq_images)
        print(_seq, eff_emb[_seq].shape)
    np.savez(EFF_CACHE, **eff_emb)

assert all(eff_emb[s].shape[0] == len(patients) for s in MRI_SEQUENCES)
X_RESNET_DEEP = np.concatenate([emb[s] for s in MRI_SEQUENCES], axis=1).astype(np.float32)
X_EFFNET_DEEP = np.concatenate([eff_emb[s] for s in MRI_SEQUENCES], axis=1).astype(np.float32)
print("ResNet50 features:", X_RESNET_DEEP.shape)
print("EfficientNet-B0 features:", X_EFFNET_DEEP.shape)
print("Device for training heads:", DEVICE_DEEP)


In [ ]:
# ================= TRAINABLE HEAD ON FROZEN CNN FEATURES =================
# Only the linear classification head is trained; the CNN backbones remain frozen.
import random
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler

DEEP_HEAD_EPOCHS = 80
DEEP_HEAD_LR = 1e-3
DEEP_HEAD_WEIGHT_DECAY = 1e-2
DEEP_HEAD_BATCH_SIZE = 64

def _set_deep_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def _fit_linear_head_all_probs(X_train, y_train, X_test_list, n_classes, seed):
    """Fit a trainable linear head on frozen CNN features and return probabilities for each test matrix."""
    _set_deep_seed(seed)
    scaler = StandardScaler().fit(np.asarray(X_train, dtype=np.float32))
    Xtr = scaler.transform(np.asarray(X_train, dtype=np.float32)).astype(np.float32)
    Xtests = [scaler.transform(np.asarray(x, dtype=np.float32)).astype(np.float32) for x in X_test_list]

    xtr = torch.from_numpy(Xtr)
    ytr = torch.from_numpy(np.asarray(y_train, dtype=np.int64))
    counts = np.bincount(np.asarray(y_train, dtype=int), minlength=n_classes).astype(np.float32)
    weights = len(y_train) / (n_classes * np.maximum(counts, 1.0))

    head = nn.Sequential(nn.Dropout(p=0.25), nn.Linear(Xtr.shape[1], n_classes)).to(DEVICE_DEEP)
    loss_fn = nn.CrossEntropyLoss(
        weight=torch.tensor(weights, dtype=torch.float32, device=DEVICE_DEEP)
    )
    optimizer = torch.optim.AdamW(
        head.parameters(), lr=DEEP_HEAD_LR, weight_decay=DEEP_HEAD_WEIGHT_DECAY
    )
    ds_train = TensorDataset(xtr, ytr)
    loader = DataLoader(
        ds_train, batch_size=min(DEEP_HEAD_BATCH_SIZE, len(ds_train)), shuffle=True,
        generator=torch.Generator().manual_seed(seed)
    )

    for _epoch in range(DEEP_HEAD_EPOCHS):
        head.train()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE_DEEP), yb.to(DEVICE_DEEP)
            optimizer.zero_grad()
            loss = loss_fn(head(xb), yb)
            loss.backward()
            optimizer.step()

    head.eval()
    out = []
    with torch.no_grad():
        for x in Xtests:
            probs = torch.softmax(head(torch.from_numpy(x).to(DEVICE_DEEP)), dim=1)
            out.append(probs.cpu().numpy())
    return out

def _fit_deep_models_for_fold(X_cnn, backbone_name, y, class_names, seed, fold, tr, te, task_id):
    """Train baseline and missingness-aware heads separately, matching the traditional experiment design."""
    n_classes = len(class_names)
    stats = clinical_stats(tr)
    X_train_img = X_cnn[tr]
    y_train = y[tr].copy()

    # Complete-clinical training set for the standard baseline (Parts 2 and 3).
    C_train_complete = clinical_features(age_all[tr], sex_all[tr], stats)
    X_fusion_train_complete = np.hstack([X_train_img, C_train_complete]).astype(np.float32)

    # Missingness-aware training set for Part 4. Keep original complete rows and add a
    # second copy with randomly hidden clinical values only when TRAIN_WITH_MISSING=True.
    if TRAIN_WITH_MISSING:
        age_aug, sex_aug = hide_randomly(
            age_all[tr], sex_all[tr],
            np.random.default_rng(seed * 10000 + fold * 101 + task_id), p=0.4
        )
        C_aug = clinical_features(age_aug, sex_aug, stats)
        X_fusion_train_missing = np.vstack([
            X_fusion_train_complete,
            np.hstack([X_train_img, C_aug]).astype(np.float32)
        ])
        y_train_missing = np.concatenate([y_train, y_train])
    else:
        X_fusion_train_missing = X_fusion_train_complete
        y_train_missing = y_train

    # Build all test matrices. The same hidden patients are used across the two backbones.
    scenario_data = {}
    X_fusion_test_list = []
    for scenario_idx, scenario in enumerate(MISSING_SCENARIOS):
        rng = np.random.default_rng(seed * 1000 + fold * 10 + scenario_idx)
        age_test, sex_test = hide_clinical(age_all[te], sex_all[te], scenario, rng)
        C_test = clinical_features(age_test, sex_test, stats)
        scenario_data[scenario] = (age_test, sex_test)
        X_fusion_test_list.append(np.hstack([X_cnn[te], C_test]).astype(np.float32))

    seed_base = seed * 100000 + fold * 1000 + task_id * 100

    # MRI-only baseline and missingness reference: clinical masking cannot change its input.
    p_mri = _fit_linear_head_all_probs(
        X_train_img, y_train, [X_cnn[te]], n_classes, seed_base + 1
    )[0]

    # Standard multimodal baseline: train only on complete clinical rows and test at 0% missing.
    p_fusion_baseline = _fit_linear_head_all_probs(
        X_fusion_train_complete, y_train, [X_fusion_test_list[0]], n_classes, seed_base + 2
    )[0]

    # Part 4 model: fit with the selected missingness-training strategy and test all scenarios.
    p_fusion_missing_list = _fit_linear_head_all_probs(
        X_fusion_train_missing, y_train_missing, X_fusion_test_list, n_classes, seed_base + 3
    )

    baseline_rows = [
        score(f"Frozen {backbone_name} + Linear Head - MRI", seed, fold,
              y[te], p_mri, class_names),
        score(f"Frozen {backbone_name} + Linear Head - MRI+Clinical", seed, fold,
              y[te], p_fusion_baseline, class_names)
    ]

    missing_rows = []
    for scenario, p_fusion in zip(MISSING_SCENARIOS, p_fusion_missing_list):
        missing_rows.append(score(
            f"Frozen {backbone_name} + Linear Head - MRI", seed, fold, y[te], p_mri,
            class_names, scenario=scenario
        ))
        missing_rows.append(score(
            f"Frozen {backbone_name} + Linear Head - MRI+Clinical", seed, fold, y[te],
            p_fusion, class_names, scenario=scenario
        ))
    return baseline_rows, missing_rows

def run_frozen_backbone_experiments(y, class_names, task_id):
    """Run both frozen backbones using the same patient-level folds and missingness scenarios."""
    baseline_rows, missing_rows = [], []
    backbones = [
        ("ResNet50", X_RESNET_DEEP),
        ("EfficientNet-B0", X_EFFNET_DEEP)
    ]
    for seed, fold, tr, te in get_folds(y):
        for backbone_name, X_cnn in backbones:
            b_rows, m_rows = _fit_deep_models_for_fold(
                X_cnn, backbone_name, y, class_names, seed, fold, tr, te, task_id
            )
            baseline_rows.extend(b_rows)
            missing_rows.extend(m_rows)
        print(f"Finished task={task_id}, seed={seed}, fold={fold + 1}/{N_FOLDS}")
    baseline_df = pd.DataFrame(baseline_rows)
    missing_df = pd.DataFrame(missing_rows)
    missing_df["scenario"] = pd.Categorical(missing_df["scenario"], MISSING_SCENARIOS, ordered=True)
    return baseline_df, missing_df


In [ ]:
# ================= RUN FROZEN RESNET50 AND EFFICIENTNET-B0 =================
# This runs binary and three-class tasks, then saves fold-level and summarized results.
deep_binary_base, deep_binary_missing = run_frozen_backbone_experiments(
    y_bin, names_bin, task_id=1
)
deep_three_base, deep_three_missing = run_frozen_backbone_experiments(
    y_3c, names_3c, task_id=2
)

def deep_summary_table(df, group_cols):
    metrics = ["mcc", "balanced_acc", "macro_f1", "auc", "accuracy"]
    metrics += [c for c in df.columns if c.startswith("recall ")]
    metrics = [c for c in metrics if c in df.columns]
    mean = df.groupby(group_cols, observed=True)[metrics].mean()
    sd = df.groupby(group_cols, observed=True)[metrics].std()
    out = mean.copy().astype(object)
    for metric in metrics:
        out[metric] = [f"{m:.3f} ± {s:.3f}" for m, s in zip(mean[metric], sd[metric])]
    return out

print("BINARY — frozen CNN feature extractors + trained linear heads")
display(deep_summary_table(deep_binary_base, ["model"]))
print("THREE-CLASS — frozen CNN feature extractors + trained linear heads")
display(deep_summary_table(deep_three_base, ["model"]))
print("BINARY — missing clinical data (MCC)")
display(deep_summary_table(deep_binary_missing, ["model", "scenario"])[["mcc", "balanced_acc"]])
print("THREE-CLASS — missing clinical data (Macro-F1)")
display(deep_summary_table(deep_three_missing, ["model", "scenario"])[["macro_f1", "balanced_acc"]])

with pd.ExcelWriter(f"frozen_cnn_results_{TAG}.xlsx") as writer:
    deep_summary_table(deep_binary_base, ["model"]).to_excel(writer, sheet_name="Binary baseline")
    deep_summary_table(deep_binary_missing, ["model", "scenario"]).to_excel(writer, sheet_name="Binary missing")
    deep_summary_table(deep_three_base, ["model"]).to_excel(writer, sheet_name="3class baseline")
    deep_summary_table(deep_three_missing, ["model", "scenario"]).to_excel(writer, sheet_name="3class missing")
    deep_binary_base.to_excel(writer, sheet_name="Binary raw folds", index=False)
    deep_binary_missing.to_excel(writer, sheet_name="Binary missing raw", index=False)
    deep_three_base.to_excel(writer, sheet_name="3class raw folds", index=False)
    deep_three_missing.to_excel(writer, sheet_name="3class missing raw", index=False)

report_lines = [
    "# Frozen ResNet50 and EfficientNet-B0 results",
    "",
    f"- Dataset: {len(patients)} patients from UCSF-PDGM.",
    "- Backbones: ImageNet-pretrained ResNet50 and EfficientNet-B0; both frozen during downstream training.",
    "- Trainable component: one linear classification head with dropout, class-weighted cross-entropy and AdamW.",
    f"- Training with simulated missing clinical values: {TRAIN_WITH_MISSING}.",
    f"- Evaluation: {N_FOLDS}-fold patient-level CV, seeds {SEEDS}; results are mean ± standard deviation across folds and seeds.",
    "- Binary primary metric: MCC. Three-class primary metric: Macro-F1.",
    "- MRI-only and MRI+Clinical are evaluated; Clinical-only is not applicable to image backbones.",
    "",
    "## Binary baseline results",
    "",
    deep_summary_table(deep_binary_base, ["model"]).to_string(),
    "",
    "## Binary missing-clinical results",
    "",
    deep_summary_table(deep_binary_missing, ["model", "scenario"])[["mcc", "balanced_acc"]].to_string(),
    "",
    "## Three-class baseline results",
    "",
    deep_summary_table(deep_three_base, ["model"]).to_string(),
    "",
    "## Three-class missing-clinical results",
    "",
    deep_summary_table(deep_three_missing, ["model", "scenario"])[["macro_f1", "balanced_acc"]].to_string(),
    "",
    "## Interpretation limits",
    "",
    "- These are new experiments and should be compared with traditional models only after this notebook has completed successfully.",
    "- The backbones are frozen; only the linear classification head is trained.",
    "- Missing clinical data is simulated, not real-world clinical missingness.",
    "- Grade 2 and Grade 3 have relatively few patients, so three-class metrics and per-class recall can be unstable.",
    "- No significance tests are performed in this notebook.",
]
with open(f"frozen_cnn_report_{TAG}.md", "w", encoding="utf-8") as f:
    f.write("\n".join(report_lines))

print(f"Saved frozen_cnn_results_{TAG}.xlsx")
print(f"Saved frozen_cnn_report_{TAG}.md")
